In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('reddit_posts_processed_16Mar.csv')

In [ ]:
df.head(20)

## Model

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
# Fill missing values in the Selftext column with an empty string
df["Selftext"] = df["Selftext"].fillna("")

In [ ]:
# Function to preprocess text
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters
    words = text.split()  # Tokenization
    return " ".join(words)  # Return cleaned text

In [ ]:
# Apply text preprocessing
df["Processed_Text"] = df["Selftext"].apply(preprocess_text)

In [ ]:
# Use the length of the processed text as a proxy for severity
df["Selftext_Length"] = df["Selftext"].apply(len)

In [ ]:
# Fill any NaN values in the target column
df["Selftext_Length"] = df["Selftext_Length"].fillna(0)

In [ ]:
# Train/Test Split for ML Model
X_train, X_test, y_train, y_test = train_test_split(df["Processed_Text"], df["Selftext_Length"], test_size=0.3, random_state=42)

In [ ]:
# Ensure no NaN values in y_train
y_train = y_train.fillna(0)

In [ ]:
# Create TF-IDF + Logistic Regression pipeline
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=200))

In [ ]:
# Train model on predicting severity score (using selftext length as a proxy)
model.fit(X_train, y_train)

In [ ]:
# Predict severity scores for all data
df["Predicted Severity Score"] = model.predict(df["Processed_Text"])

In [ ]:
# Normalize severity scores to range [1, 10]
df["Predicted Severity Score"] = df["Predicted Severity Score"].apply(lambda x: max(1, min(10, round(x / df["Predicted Severity Score"].max() * 10))))

In [ ]:
# Save results
output_csv_path = "severity_test_with_predicted_severity.csv"
df.to_csv(output_csv_path, index=False)

print(f"Processed file saved as {output_csv_path}")

In [ ]:
df_new = pd.read_csv('/content/severity_test_with_nlp_predictions.csv')

In [ ]:
df_new = df_new.drop('Assigned Disorder',axis=1)

In [ ]:
df_new[df_new['Subreddit'] == 'depression'].head(10)

In [ ]:
df_new[df_new['Subreddit'] == 'ADHD'].head(10)